# Análisis de Sentimientos: De Básico a Avanzado

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leoe21/machine_learning_fundamentals/blob/main/03_unidad/RNN_Text/Sesion1/7-sentiment-analysis.ipynb)

---

## 📚 Introducción

Este notebook presenta un recorrido completo por diferentes técnicas de análisis de sentimientos, desde métodos básicos hasta modelos avanzados pre-entrenados. A través de tres ejercicios prácticos, aprenderás a:

- **Aplicar VADER** (Valence Aware Dictionary and sEntiment Reasoner) para análisis de sentimientos en inglés
- **Construir un modelo de Machine Learning** desde cero usando Naive Bayes y técnicas de NLP básicas
- **Comparar con modelos pre-entrenados** como pysentimiento para español
- **Entender las ventajas y desventajas** de cada enfoque

### 🎯 Objetivos

1. ✅ Preprocesar texto usando SpaCy (tokenización, lemmatización, filtrado)
2. ✅ Extraer características usando TF-IDF
3. ✅ Entrenar y evaluar modelos de clasificación de sentimientos
4. ✅ Manejar datasets desbalanceados usando técnicas de resampling
5. ✅ Realizar ajuste de hiperparámetros
6. ✅ Comparar diferentes enfoques de análisis de sentimientos
7. ✅ Interpretar métricas de evaluación (accuracy, precision, recall, F1-score)

### 📋 Estructura del Notebook

Este notebook está dividido en **4 partes principales**:

- **Parte I**: Análisis de Sentimientos con VADER (Inglés)
  - Dataset de reseñas de películas
  - Uso de modelo pre-entrenado sin entrenamiento
  
- **Parte II**: Análisis de Sentimientos con Machine Learning (Español)
  - Dataset de tweets sobre Inteligencia Artificial
  - Preprocesamiento con SpaCy
  - Entrenamiento de modelo Naive Bayes desde cero
  - Balanceo de datos y ajuste de hiperparámetros
  
- **Parte III**: Comparación con pysentimiento
  - Uso de modelo pre-entrenado para español
  - Evaluación comparativa
  
- **Parte IV**: Conclusiones y Comparación Final
  - Análisis de ventajas y desventajas
  - Guía de cuándo usar cada enfoque

### 📚 Referencias

* [Natural Language Processing in Action](https://www.manning.com/books/natural-language-processing-in-action)
* [pysentimiento Documentation](https://github.com/pysentimiento/pysentimiento)
* [SpaCy Documentation](https://spacy.io/)

---

## 📑 Tabla de Contenidos

### 🔧 Configuración Inicial
- Instalación de dependencias
- Carga de datasets

### 📖 [Parte I: Análisis de Sentimientos con VADER (Inglés)](#parte-i)

### 🤖 [Parte II: Análisis de Sentimientos con Machine Learning (Español)](#parte-ii)

### 🚀 [Parte III: Comparación con pysentimiento](#parte-iii)

### 📊 [Parte IV: Conclusiones y Comparación Final](#parte-iv)


---


In [2]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

In [ ]:
!test '{IN_COLAB}' = 'True' && wget  https://github.com/leoe21/machine_learning_fundamentals/raw/main/requirements.txt && pip install -r requirements.txt

---

<a id="parte-i"></a>
# 📖 Parte I: Análisis de Sentimientos con VADER (Inglés)

En esta primera parte, utilizaremos **VADER** (Valence Aware Dictionary and sEntiment Reasoner), un modelo pre-entrenado de NLTK diseñado específicamente para análisis de sentimientos en texto en inglés. VADER es especialmente útil para texto de redes sociales y no requiere entrenamiento.

**Dataset**: Reseñas de películas en inglés  
**Objetivo**: Clasificar reseñas como positivas o negativas

---


In [ ]:
!test '{IN_COLAB}' = 'True' && wget  https://github.com/leoe21/machine_learning_fundamentals/raw/main/03_unidad/RNN_Text/Sesion1/moviereviews.tsv

Empecemos por cargar el dataset:

In [1]:
import pandas as pd
import numpy as np

reviews = pd.read_csv('./moviereviews.tsv', sep='\t')
reviews.head()

,label,review
0,neg,how do films like mouse hunt get into theatres...
1,neg,some talented actresses are blessed with a dem...
2,pos,this has been an extraordinary year for austra...
3,pos,according to hollywood movies made in last few...
4,neg,my first press screening of 1998 and already i...


Luego, hagamos algo de limpieza, vamos a remover nulos y valores vacíos:

In [2]:
reviews.dropna(inplace=True)
reviews.review = reviews.review.apply(lambda r: r.strip())
blanks = reviews[reviews.review == ''].index
reviews.drop(blanks, inplace=True)

In [3]:
reviews[reviews.review == ''].index

Index([], dtype='int64')

In [4]:
reviews.label.value_counts()

label
neg    969
pos    969
Name: count, dtype: int64

Tenemos un dataset balanceado de casi mil ejemplares por cada clase.

Para hacer las cosas simples, vamos a utilizar un VADER para computar el puntaje de positivo o negativo. Este modelo ya viene implementado dentro de NLTK.

In [5]:
import nltk
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Usuario\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

In [6]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sid = SentimentIntensityAnalyzer()
reviews['scores'] = reviews.review.apply(lambda r: sid.polarity_scores(r))
reviews.head()

,label,review,scores
0,neg,how do films like mouse hunt get into theatres...,"{'neg': 0.121, 'neu': 0.778, 'pos': 0.101, 'co..."
1,neg,some talented actresses are blessed with a dem...,"{'neg': 0.12, 'neu': 0.775, 'pos': 0.105, 'com..."
2,pos,this has been an extraordinary year for austra...,"{'neg': 0.068, 'neu': 0.781, 'pos': 0.15, 'com..."
3,pos,according to hollywood movies made in last few...,"{'neg': 0.071, 'neu': 0.782, 'pos': 0.147, 'co..."
4,neg,my first press screening of 1998 and already i...,"{'neg': 0.091, 'neu': 0.817, 'pos': 0.093, 'co..."


Con estos puntajes ahora podemos convertir el resultado en una etiqueta de predicción:

In [7]:
reviews['compound'] = reviews.scores.apply(lambda s: s['compound'])    
reviews['prediction'] = reviews['compound'].apply(lambda c: 'pos' if c > 0 else 'neg')
reviews.head()

,label,review,scores,compound,prediction
0,neg,how do films like mouse hunt get into theatres...,"{'neg': 0.121, 'neu': 0.778, 'pos': 0.101, 'co...",-0.9125,neg
1,neg,some talented actresses are blessed with a dem...,"{'neg': 0.12, 'neu': 0.775, 'pos': 0.105, 'com...",-0.8618,neg
2,pos,this has been an extraordinary year for austra...,"{'neg': 0.068, 'neu': 0.781, 'pos': 0.15, 'com...",0.9951,pos
3,pos,according to hollywood movies made in last few...,"{'neg': 0.071, 'neu': 0.782, 'pos': 0.147, 'co...",0.9972,pos
4,neg,my first press screening of 1998 and already i...,"{'neg': 0.091, 'neu': 0.817, 'pos': 0.093, 'co...",-0.2484,neg


Y finalmente computar unas cuantas métricas de calidad del modelo:

In [8]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_true = reviews.label.values
y_pred = reviews.prediction.values

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)
cr = classification_report(y_true, y_pred)


print(f"Accuracy:\n{acc}\n")
print(f"Classification Report:\n{cr}")
print(f"Confusion Matrix:\n{cm}")

Accuracy:
0.6357069143446853

Classification Report:
              precision    recall  f1-score   support

         neg       0.72      0.44      0.55       969
         pos       0.60      0.83      0.70       969

    accuracy                           0.64      1938
   macro avg       0.66      0.64      0.62      1938
weighted avg       0.66      0.64      0.62      1938

Confusion Matrix:
[[427 542]
 [164 805]]


---

<a id="parte-ii"></a>
# 🤖 Parte II: Análisis de Sentimientos con Machine Learning (Español)

En esta segunda parte, construiremos un modelo de Machine Learning desde cero usando técnicas aprendidas en los notebooks anteriores. Aplicaremos preprocesamiento con SpaCy, extracción de características con TF-IDF, y entrenaremos un clasificador Naive Bayes.

**Dataset**: 4,038 tweets en español sobre Inteligencia Artificial  
**Objetivo**: Clasificar tweets como positivos, negativos o neutros  
**Técnicas**: Tokenización, lemmatización, TF-IDF, Naive Bayes, balanceo de datos

**Fuente del dataset**: [Zenodo - IA Tweets Dataset](https://zenodo.org/records/10821485)

---


La correctitud no es la mejor, aún podemos hacerlo mucho mejor que la línea base (50%). Parece que tenemos problemas con las etiquetas negativas!

# Ejercicio con 4.038 tweets en español, relacionados discusiones acerca de la Inteligencia Artificial 

#### Fuente: (https://zenodo.org/records/10821485?utm_source=chatgpt.com)

## Explorar el dataset

In [9]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Cargar dataset
print("Cargando dataset...")
df = pd.read_csv('./ia_tweets.csv')

# Explorar estructura
print("\n" + "="*60)
print("INFORMACIÓN DEL DATASET")
print("="*60)
print(f"\n📊 Forma del dataset: {df.shape}")
print(f"   Filas: {df.shape[0]}")
print(f"   Columnas: {df.shape[1]}")

print(f"\n📋 Nombres de columnas:")
print(df.columns.tolist())

print(f"\n🔍 Primeras 5 filas:")
print(df.head())

print(f"\n📊 Información general:")
print(df.info())

print(f"\n📈 Valores nulos por columna:")
print(df.isnull().sum())

# Si hay columna de sentimiento, ver distribución
if 'sentiment' in df.columns or 'label' in df.columns or 'sentimiento' in df.columns:
    col_sent = [c for c in df.columns if 'sent' in c.lower() or 'label' in c.lower()][0]
    print(f"\n📊 Distribución de sentimientos:")
    print(df[col_sent].value_counts())

Cargando dataset...

INFORMACIÓN DEL DATASET

📊 Forma del dataset: (4038, 14)
   Filas: 4038
   Columnas: 14

📋 Nombres de columnas:
['ID', 'text', 'polarity', 'favorite_count', 'retweet_count', 'user_followers_count', 'user_friends_count', 'user_favourites_count', 'user_statuses_count', 'user_verified', 'user_has_extended_profile', 'user_is_translator', 'user_protected', 'user_default_profile']

🔍 Primeras 5 filas:
   ID                                               text polarity  \
0   0  Comentaba en una charla sobre IA que el proble...        N   
1   1  @alvaropons Eso es imposible. Y sí, va a gener...        N   
2   2  @alvaropons Lo disruptivo es que tras años de ...      NEU   
3   3  @alvaropons ¿"Prohibición del uso comercial de...        N   
4   4  @santoroydonoso Cómo se está estudiando prohib...      NEU   

   favorite_count  retweet_count  user_followers_count  user_friends_count  \
0              21              7                 15936                2395   
1        

## Explorar la columna de polaridad

In [10]:
# Explorar la columna de polaridad
print("="*60)
print("ANÁLISIS DE POLARIDAD")
print("="*60)

print("\n📊 Valores únicos en 'polarity':")
print(df['polarity'].unique())

print("\n📊 Distribución de polaridad:")
print(df['polarity'].value_counts())

print("\n📊 Distribución porcentual:")
print(df['polarity'].value_counts(normalize=True) * 100)

# Ver algunos ejemplos de cada clase
print("\n" + "="*60)
print("EJEMPLOS POR CLASE")
print("="*60)

for polarity in df['polarity'].unique():
    print(f"\n{polarity}:")
    ejemplos = df[df['polarity'] == polarity]['text'].head(2)
    for i, texto in enumerate(ejemplos, 1):
        print(f"  {i}. {texto[:100]}...")

ANÁLISIS DE POLARIDAD

📊 Valores únicos en 'polarity':
['N' 'NEU' 'P']

📊 Distribución de polaridad:
polarity
NEU    2689
P       757
N       592
Name: count, dtype: int64

📊 Distribución porcentual:
polarity
NEU    66.592372
P      18.746904
N      14.660723
Name: proportion, dtype: float64

EJEMPLOS POR CLASE

N:
  1. Comentaba en una charla sobre IA que el problema no estaba en el debate sobre derechos de autor, sin...
  2. @alvaropons Eso es imposible. Y sí, va a generar un montón de pérdida de puestos de trabajo y no sol...

NEU:
  1. @alvaropons Lo disruptivo es que tras años de decir que la automatización iba a acabar con los puest...
  2. @santoroydonoso Cómo se está estudiando prohibir chatGPT en Europa? No te digo que no sea difícil de...

P:
  1. 8/ Conclusión: La inteligencia artificial ha llegado para quedarse, y su impacto en la experiencia d...
  2. 9/ ¿Te gustó este hilo? ¡Comparte y sígueme para más contenidos sobre #InteligenciaArtificial y #Exp...


## Preparar y limpiar datos

In [11]:
# Preparar datos para el análisis
print("="*60)
print("PREPARACIÓN DE DATOS")
print("="*60)

# Crear copia del dataframe
reviews = df.copy()

# Mapear etiquetas a formato estándar (minúsculas)
label_mapping = {
    'N': 'neg',
    'NEU': 'neu',
    'P': 'pos'
}

reviews['label'] = reviews['polarity'].map(label_mapping)

# Verificar mapeo
print("\n📊 Distribución después del mapeo:")
print(reviews['label'].value_counts())
print(f"\nTotal: {len(reviews)} tweets")

# Limpieza básica
print("\n🧹 Limpiando datos...")

# Eliminar espacios en blanco
reviews['text'] = reviews['text'].apply(lambda r: str(r).strip())

# Eliminar filas vacías (si las hay)
blanks = reviews[reviews['text'] == ''].index
if len(blanks) > 0:
    reviews.drop(blanks, inplace=True)
    print(f"   Eliminadas {len(blanks)} filas vacías")

print(f"✓ Datos limpios: {len(reviews)} tweets")

# Verificar balance final
print(f"\n📊 Distribución final:")
print(reviews['label'].value_counts())
print(f"\n📊 Distribución porcentual:")
print(reviews['label'].value_counts(normalize=True) * 100)

PREPARACIÓN DE DATOS

📊 Distribución después del mapeo:
label
neu    2689
pos     757
neg     592
Name: count, dtype: int64

Total: 4038 tweets

🧹 Limpiando datos...
✓ Datos limpios: 4038 tweets

📊 Distribución final:
label
neu    2689
pos     757
neg     592
Name: count, dtype: int64

📊 Distribución porcentual:
label
neu    66.592372
pos    18.746904
neg    14.660723
Name: proportion, dtype: float64


## Cargar SpaCy y función de preprocesamiento

In [12]:
import spacy
from collections import Counter

# Cargar modelo SpaCy en español
print("Cargando modelo SpaCy en español...")
nlp = spacy.load('es_core_news_lg')
print("✓ Modelo cargado")

# Función de preprocesamiento
def preprocess_text(text):
    """
    Preprocesa texto usando:
    - Tokenización (Notebook 2)
    - Lemmatization (Notebook 4)
    - Filtrado de stop words
    """
    doc = nlp(str(text))
    
    # Extraer lemas, filtrar stop words y puntuación
    tokens = [
        token.lemma_.lower() 
        for token in doc 
        if token.is_alpha and not token.is_stop and len(token.text) > 2
    ]
    
    return ' '.join(tokens)

# Probar con un ejemplo
print("\n🔍 Ejemplo de preprocesamiento:")
ejemplo = reviews.iloc[0]['text']
print(f"Original: {ejemplo[:100]}...")
print(f"Procesado: {preprocess_text(ejemplo)[:100]}...")

Cargando modelo SpaCy en español...
✓ Modelo cargado

🔍 Ejemplo de preprocesamiento:
Original: Comentaba en una charla sobre IA que el problema no estaba en el debate sobre derechos de autor, sin...
Procesado: comentar charla problema debate derecho autor empresa ver oportunidad abaratar coste convencido proh...


## Preprocesar todos los textos


In [13]:
# Preprocesar todos los textos
print("="*60)
print("PREPROCESAMIENTO CON SPA CY")
print("="*60)

# Aplicar preprocesamiento a todos los textos
reviews['text_processed'] = reviews['text'].apply(preprocess_text)

print(f"✓ Preprocesamiento completado")
print(f"\nEjemplos:")
for i in range(3):
    print(f"\n{i+1}. Original: {reviews.iloc[i]['text'][:80]}...")
    print(f"   Procesado: {reviews.iloc[i]['text_processed'][:80]}...")

PREPROCESAMIENTO CON SPA CY
✓ Preprocesamiento completado

Ejemplos:

1. Original: Comentaba en una charla sobre IA que el problema no estaba en el debate sobre de...
   Procesado: comentar charla problema debate derecho autor empresa ver oportunidad abaratar c...

2. Original: @alvaropons Eso es imposible. Y sí, va a generar un montón de pérdida de puestos...
   Procesado: imposible generar montón pérdida puesto trabajo ilustración chat gpt escribir có...

3. Original: @alvaropons Lo disruptivo es que tras años de decir que la automatización iba a ...
   Procesado: disruptivo año automatización ir acabar puesto trabajo cualificado llevar él tip...


## Análisis exploratorio de palabras

In [14]:
# Análisis exploratorio
print("="*60)
print("ANÁLISIS EXPLORATORIO DE PALABRAS")
print("="*60)

# Palabras más frecuentes por sentimiento
for label in ['pos', 'neg', 'neu']:
    label_reviews = reviews[reviews.label == label]['text_processed']
    if len(label_reviews) > 0:
        words = ' '.join(label_reviews).split()
        word_freq = Counter(words)
        print(f"\n📊 Top 10 palabras en tweets {label.upper()}:")
        for word, count in word_freq.most_common(10):
            print(f"   {word:20} → {count:4} veces")

ANÁLISIS EXPLORATORIO DE PALABRAS

📊 Top 10 palabras en tweets POS:
   chatgpt              →  189 veces
   inteligencia         →  184 veces
   artificial           →  180 veces
   él                   →   83 veces
   análisis             →   66 veces
   resultado            →   62 veces
   esperar              →   61 veces
   mejorar              →   57 veces
   tecnología           →   55 veces
   mejora               →   55 veces

📊 Top 10 palabras en tweets NEG:
   inteligencia         →  115 veces
   artificial           →  108 veces
   él                   →   92 veces
   chatgpt              →   85 veces
   empresa              →   43 veces
   hablar               →   29 veces
   ver                  →   28 veces
   dato                 →   28 veces
   pedir                →   28 veces
   humano               →   28 veces

📊 Top 10 palabras en tweets NEU:
   inteligencia         →  575 veces
   artificial           →  557 veces
   chatgpt              →  537 veces
   él        

## División Train/Validation/Test

In [15]:
from sklearn.model_selection import train_test_split

# ============================================
# DIVISIÓN EN 3 CONJUNTOS: TRAIN / VALIDATION / TEST
# ============================================

X = reviews['text_processed']
y = reviews['label']

# Primero dividir en Train+Validation (80%) y Test (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y,
    test_size=0.2,  # 20% para test
    random_state=42,
    stratify=y  # Mantener proporción de clases
)

# Luego dividir Train+Validation en Train (70%) y Validation (10% del total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,  # 12.5% de X_temp = 10% del total
    random_state=42,
    stratify=y_temp
)

print("="*60)
print("DIVISIÓN DE DATOS")
print("="*60)
print(f"\n📊 Distribución:")
print(f"   Train:      {len(X_train):5} tweets ({len(X_train)/len(reviews)*100:.1f}%)")
print(f"   Validation: {len(X_val):5} tweets ({len(X_val)/len(reviews)*100:.1f}%)")
print(f"   Test:       {len(X_test):5} tweets ({len(X_test)/len(reviews)*100:.1f}%)")
print(f"   Total:      {len(reviews):5} tweets")

print(f"\n📊 Distribución de clases en Train:")
print(y_train.value_counts())
print(f"\n📊 Distribución de clases en Validation:")
print(y_val.value_counts())
print(f"\n📊 Distribución de clases en Test:")
print(y_test.value_counts())

DIVISIÓN DE DATOS

📊 Distribución:
   Train:       2826 tweets (70.0%)
   Validation:   404 tweets (10.0%)
   Test:         808 tweets (20.0%)
   Total:       4038 tweets

📊 Distribución de clases en Train:
label
neu    1882
pos     529
neg     415
Name: count, dtype: int64

📊 Distribución de clases en Validation:
label
neu    269
pos     76
neg     59
Name: count, dtype: int64

📊 Distribución de clases en Test:
label
neu    538
pos    152
neg    118
Name: count, dtype: int64


## Crear características TF-IDF

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Crear características usando TF-IDF
print("="*60)
print("CREACIÓN DE CARACTERÍSTICAS TF-IDF")
print("="*60)
print("(Ajustando vectorizador solo con datos de entrenamiento)")

vectorizer = TfidfVectorizer(
    max_features=1000,  # Usar las 1000 palabras más importantes
    ngram_range=(1, 2),  # Incluir palabras individuales y bigramas
    min_df=2,  # Palabra debe aparecer en al menos 2 documentos
    max_df=0.8  # Palabra no debe aparecer en más del 80% de documentos
)

# Ajustar SOLO con datos de entrenamiento
X_train_vec = vectorizer.fit_transform(X_train)

# Transformar validation y test (sin ajustar)
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print(f"✓ Características creadas")
print(f"   Train:      {X_train_vec.shape}")
print(f"   Validation: {X_val_vec.shape}")
print(f"   Test:       {X_test_vec.shape}")
print(f"   Vocabulario: {len(vectorizer.vocabulary_)} palabras")

CREACIÓN DE CARACTERÍSTICAS TF-IDF
(Ajustando vectorizador solo con datos de entrenamiento)
✓ Características creadas
   Train:      (2826, 1000)
   Validation: (404, 1000)
   Test:       (808, 1000)
   Vocabulario: 1000 palabras


## Entrenar modelo Naive Bayes

In [17]:
from sklearn.naive_bayes import MultinomialNB

# Entrenar clasificador Naive Bayes
print("="*60)
print("ENTRENAMIENTO DEL MODELO")
print("="*60)

clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

print("✓ Modelo entrenado")

ENTRENAMIENTO DEL MODELO
✓ Modelo entrenado


## Evaluar en Validation

In [18]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Evaluar en Validation
print("="*60)
print("EVALUACIÓN EN VALIDATION SET")
print("="*60)

y_val_pred = clf.predict(X_val_vec)

acc_val = accuracy_score(y_val, y_val_pred)
cm_val = confusion_matrix(y_val, y_val_pred)  # ← CORREGIDO: era y_pred, ahora es y_val_pred
cr_val = classification_report(y_val, y_val_pred)

print(f"\n📊 Accuracy en Validation: {acc_val:.4f} ({acc_val*100:.2f}%)")
print(f"\n📋 Classification Report:\n{cr_val}")
print(f"\n🔢 Confusion Matrix:\n{cm_val}")

# Guardar para comparar después
validation_accuracy = acc_val

EVALUACIÓN EN VALIDATION SET

📊 Accuracy en Validation: 0.6980 (69.80%)

📋 Classification Report:
              precision    recall  f1-score   support

         neg       0.67      0.03      0.06        59
         neu       0.69      0.99      0.81       269
         pos       0.93      0.17      0.29        76

    accuracy                           0.70       404
   macro avg       0.76      0.40      0.39       404
weighted avg       0.73      0.70      0.61       404


🔢 Confusion Matrix:
[[  2  57   0]
 [  1 267   1]
 [  0  63  13]]


El desbalanceo de la clase neutral esta tendiendo a que la predicción se incline hacia esta clase tambien. Realizaremos balanceo de clases.

In [19]:
from sklearn.utils import resample

print("="*60)
print("BALANCEO DEL DATASET")
print("="*60)

# Volver a cargar el dataset original (antes de la división)
# O usar el dataframe original 'df' que cargaste al inicio
reviews_original = df.copy()

# Mapear etiquetas
label_mapping = {'N': 'neg', 'NEU': 'neu', 'P': 'pos'}
reviews_original['label'] = reviews_original['polarity'].map(label_mapping)

# Limpiar
reviews_original['text'] = reviews_original['text'].apply(lambda r: str(r).strip())

# Preprocesar (si ya tienes la función preprocess_text definida)
print("\nPreprocesando textos...")
reviews_original['text_processed'] = reviews_original['text'].apply(preprocess_text)

# Ver distribución original
print("\n📊 Distribución ORIGINAL:")
print(reviews_original['label'].value_counts())

# Separar por clase
df_neg = reviews_original[reviews_original.label == 'neg']
df_neu = reviews_original[reviews_original.label == 'neu']
df_pos = reviews_original[reviews_original.label == 'pos']

# Determinar tamaño objetivo
# Usar un tamaño intermedio para balancear mejor
min_size = min(len(df_neg), len(df_pos))
target_size = min(700, min_size * 2)  # Usar 700 o el doble de la clase más pequeña

print(f"\n📊 Tamaños originales:")
print(f"  Negativo: {len(df_neg)}")
print(f"  Neutro: {len(df_neu)}")
print(f"  Positivo: {len(df_pos)}")
print(f"\n🎯 Tamaño objetivo por clase: {target_size}")

# Submuestrear la clase mayoritaria (neutro)
df_neu_balanced = resample(df_neu, 
                          replace=False,  # Sin reemplazo (submuestrear)
                          n_samples=target_size, 
                          random_state=42)

# Sobremuestrear las clases minoritarias
df_neg_balanced = resample(df_neg, 
                          replace=True,  # Con reemplazo (sobremuestrear)
                          n_samples=target_size, 
                          random_state=42)

df_pos_balanced = resample(df_pos, 
                          replace=True,  # Con reemplazo (sobremuestrear)
                          n_samples=target_size, 
                          random_state=42)

# Combinar
reviews_balanced = pd.concat([df_neg_balanced, df_neu_balanced, df_pos_balanced])
reviews_balanced = reviews_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n✅ Tamaños después de balanceo:")
print(f"  Negativo: {len(df_neg_balanced)}")
print(f"  Neutro: {len(df_neu_balanced)}")
print(f"  Positivo: {len(df_pos_balanced)}")
print(f"  Total: {len(reviews_balanced)}")

print(f"\n📊 Distribución balanceada:")
print(reviews_balanced['label'].value_counts())
print(f"\n📊 Distribución porcentual:")
print(reviews_balanced['label'].value_counts(normalize=True) * 100)

BALANCEO DEL DATASET

Preprocesando textos...

📊 Distribución ORIGINAL:
label
neu    2689
pos     757
neg     592
Name: count, dtype: int64

📊 Tamaños originales:
  Negativo: 592
  Neutro: 2689
  Positivo: 757

🎯 Tamaño objetivo por clase: 700

✅ Tamaños después de balanceo:
  Negativo: 700
  Neutro: 700
  Positivo: 700
  Total: 2100

📊 Distribución balanceada:
label
neu    700
neg    700
pos    700
Name: count, dtype: int64

📊 Distribución porcentual:
label
neu    33.333333
neg    33.333333
pos    33.333333
Name: proportion, dtype: float64


In [20]:
print("="*60)
print("DIVISIÓN DE DATOS BALANCEADOS")
print("="*60)

X_balanced = reviews_balanced['text_processed']
y_balanced = reviews_balanced['label']

# Primero dividir en Train+Validation (80%) y Test (20%)
X_temp, X_test, y_temp, y_test = train_test_split(
    X_balanced, y_balanced,
    test_size=0.2,
    random_state=42,
    stratify=y_balanced
)

# Luego dividir Train+Validation en Train (70%) y Validation (10% del total)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=0.125,
    random_state=42,
    stratify=y_temp
)

print(f"\n📊 Nueva distribución:")
print(f"   Train:      {len(X_train):5} tweets ({len(X_train)/len(reviews_balanced)*100:.1f}%)")
print(f"   Validation: {len(X_val):5} tweets ({len(X_val)/len(reviews_balanced)*100:.1f}%)")
print(f"   Test:       {len(X_test):5} tweets ({len(X_test)/len(reviews_balanced)*100:.1f}%)")
print(f"   Total:      {len(reviews_balanced):5} tweets")

print(f"\n📊 Distribución de clases en Train:")
print(y_train.value_counts())
print(f"\n📊 Distribución de clases en Validation:")
print(y_val.value_counts())
print(f"\n📊 Distribución de clases en Test:")
print(y_test.value_counts())

DIVISIÓN DE DATOS BALANCEADOS

📊 Nueva distribución:
   Train:       1470 tweets (70.0%)
   Validation:   210 tweets (10.0%)
   Test:         420 tweets (20.0%)
   Total:       2100 tweets

📊 Distribución de clases en Train:
label
neu    490
pos    490
neg    490
Name: count, dtype: int64

📊 Distribución de clases en Validation:
label
pos    70
neu    70
neg    70
Name: count, dtype: int64

📊 Distribución de clases en Test:
label
neu    140
neg    140
pos    140
Name: count, dtype: int64


In [21]:
print("="*60)
print("CREACIÓN DE CARACTERÍSTICAS (BALANCEADAS)")
print("="*60)

# Crear nuevo vectorizador
vectorizer = TfidfVectorizer(
    max_features=1000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.8
)

# Ajustar SOLO con datos de entrenamiento balanceados
X_train_vec = vectorizer.fit_transform(X_train)

# Transformar validation y test
X_val_vec = vectorizer.transform(X_val)
X_test_vec = vectorizer.transform(X_test)

print(f"✓ Características creadas")
print(f"   Train:      {X_train_vec.shape}")
print(f"   Validation: {X_val_vec.shape}")
print(f"   Test:       {X_test_vec.shape}")
print(f"   Vocabulario: {len(vectorizer.vocabulary_)} palabras")

CREACIÓN DE CARACTERÍSTICAS (BALANCEADAS)
✓ Características creadas
   Train:      (1470, 1000)
   Validation: (210, 1000)
   Test:       (420, 1000)
   Vocabulario: 1000 palabras


In [22]:
print("="*60)
print("ENTRENAMIENTO CON DATOS BALANCEADOS")
print("="*60)

# Entrenar nuevo modelo
clf = MultinomialNB()
clf.fit(X_train_vec, y_train)

print("✓ Modelo reentrenado con datos balanceados")

ENTRENAMIENTO CON DATOS BALANCEADOS
✓ Modelo reentrenado con datos balanceados


In [23]:
print("="*60)
print("EVALUACIÓN EN VALIDATION (BALANCEADO)")
print("="*60)

y_val_pred = clf.predict(X_val_vec)

acc_val = accuracy_score(y_val, y_val_pred)
cm_val = confusion_matrix(y_val, y_val_pred)
cr_val = classification_report(y_val, y_val_pred)

print(f"\n📊 Accuracy en Validation: {acc_val:.4f} ({acc_val*100:.2f}%)")
print(f"\n📋 Classification Report:\n{cr_val}")
print(f"\n🔢 Confusion Matrix:\n{cm_val}")

# Guardar para comparar
validation_accuracy = acc_val

print("\n" + "="*60)
print("COMPARACIÓN: ANTES vs DESPUÉS DEL BALANCEO")
print("="*60)
print("Antes del balanceo:")
print("  - Accuracy: ~69.80%")
print("  - Recall Negativo: 0.03 (muy bajo)")
print("  - Recall Positivo: 0.17 (muy bajo)")
print("  - Recall Neutro: 0.99 (muy alto)")
print("\nDespués del balanceo:")
print(f"  - Accuracy: {acc_val:.2%}")
print("  - Revisa el Classification Report arriba para ver mejoras")

EVALUACIÓN EN VALIDATION (BALANCEADO)

📊 Accuracy en Validation: 0.6381 (63.81%)

📋 Classification Report:
              precision    recall  f1-score   support

         neg       0.65      0.71      0.68        70
         neu       0.55      0.54      0.55        70
         pos       0.72      0.66      0.69        70

    accuracy                           0.64       210
   macro avg       0.64      0.64      0.64       210
weighted avg       0.64      0.64      0.64       210


🔢 Confusion Matrix:
[[50 17  3]
 [17 38 15]
 [10 14 46]]

COMPARACIÓN: ANTES vs DESPUÉS DEL BALANCEO
Antes del balanceo:
  - Accuracy: ~69.80%
  - Recall Negativo: 0.03 (muy bajo)
  - Recall Positivo: 0.17 (muy bajo)
  - Recall Neutro: 0.99 (muy alto)

Después del balanceo:
  - Accuracy: 63.81%
  - Revisa el Classification Report arriba para ver mejoras


Análisis
El modelo ahora detecta mejor las tres clases.
El accuracy bajó ligeramente, pero el rendimiento es más equilibrado.
La matriz de confusión muestra mejor distribución de errores.

## Ajuste de hiperparámetros

In [24]:
# ============================================
# AJUSTE DE HIPERPARÁMETROS
# ============================================

print("="*60)
print("AJUSTE DE HIPERPARÁMETROS")
print("="*60)

# Probar diferentes valores de alpha (suavizado de Laplace)
alphas = [0.1, 0.5, 1.0, 2.0, 5.0]
best_alpha = 1.0
best_score = validation_accuracy
best_results = {}

print("\nProbando diferentes valores de alpha:")
for alpha in alphas:
    clf_temp = MultinomialNB(alpha=alpha)
    clf_temp.fit(X_train_vec, y_train)
    y_val_pred_temp = clf_temp.predict(X_val_vec)
    score = accuracy_score(y_val, y_val_pred_temp)
    
    # Calcular métricas por clase
    cr_temp = classification_report(y_val, y_val_pred_temp, output_dict=True)
    
    print(f"\n  Alpha {alpha:4.1f}:")
    print(f"    Accuracy: {score:.4f}")
    print(f"    Recall Neg: {cr_temp['neg']['recall']:.3f}, "
          f"Neu: {cr_temp['neu']['recall']:.3f}, "
          f"Pos: {cr_temp['pos']['recall']:.3f}")
    
    if score > best_score:
        best_score = score
        best_alpha = alpha
        best_results = {
            'alpha': alpha,
            'accuracy': score,
            'report': cr_temp
        }

print(f"\n✓ Mejor alpha: {best_alpha} (Accuracy: {best_score:.4f})")

# Entrenar modelo final con mejor alpha
clf = MultinomialNB(alpha=best_alpha)
clf.fit(X_train_vec, y_train)
print("✓ Modelo final entrenado con mejor hiperparámetro")

# Reevaluar con mejor modelo
y_val_pred_final = clf.predict(X_val_vec)
acc_val_final = accuracy_score(y_val, y_val_pred_final)
cr_val_final = classification_report(y_val, y_val_pred_final)

print(f"\n📊 Accuracy final en Validation: {acc_val_final:.4f} ({acc_val_final*100:.2f}%)")
print(f"\n📋 Classification Report Final:\n{cr_val_final}")

# Actualizar validation_accuracy
validation_accuracy = acc_val_final

AJUSTE DE HIPERPARÁMETROS

Probando diferentes valores de alpha:

  Alpha  0.1:
    Accuracy: 0.6143
    Recall Neg: 0.671, Neu: 0.514, Pos: 0.657

  Alpha  0.5:
    Accuracy: 0.6381
    Recall Neg: 0.714, Neu: 0.543, Pos: 0.657

  Alpha  1.0:
    Accuracy: 0.6381
    Recall Neg: 0.714, Neu: 0.543, Pos: 0.657

  Alpha  2.0:
    Accuracy: 0.6286
    Recall Neg: 0.700, Neu: 0.514, Pos: 0.671

  Alpha  5.0:
    Accuracy: 0.6190
    Recall Neg: 0.686, Neu: 0.514, Pos: 0.657

✓ Mejor alpha: 1.0 (Accuracy: 0.6381)
✓ Modelo final entrenado con mejor hiperparámetro

📊 Accuracy final en Validation: 0.6381 (63.81%)

📋 Classification Report Final:
              precision    recall  f1-score   support

         neg       0.65      0.71      0.68        70
         neu       0.55      0.54      0.55        70
         pos       0.72      0.66      0.69        70

    accuracy                           0.64       210
   macro avg       0.64      0.64      0.64       210
weighted avg       0.64      

---

<a id="parte-iii"></a>

# 🚀 Parte III: Comparación con pysentimiento

En esta tercera parte, compararemos nuestro modelo entrenado desde cero con **pysentimiento**, un modelo pre-entrenado basado en transformers específicamente diseñado para análisis de sentimientos en español. Esto nos permitirá entender las diferencias entre modelos básicos y modelos avanzados.

**Objetivo**: Comparar Naive Bayes vs pysentimiento en el mismo dataset de test

---


In [25]:
# ============================================
# EVALUACIÓN FINAL EN TEST SET
# ============================================

print("="*60)
print("EVALUACIÓN FINAL EN TEST SET")
print("="*60)
print("(Esta es la evaluación final, no debe usarse para ajustar el modelo)")

y_test_pred = clf.predict(X_test_vec)
y_test_proba = clf.predict_proba(X_test_vec)

acc_test = accuracy_score(y_test, y_test_pred)
cm_test = confusion_matrix(y_test, y_test_pred)
cr_test = classification_report(y_test, y_test_pred)

print(f"\n📊 Accuracy en Test: {acc_test:.4f} ({acc_test*100:.2f}%)")
print(f"\n📋 Classification Report:\n{cr_test}")
print(f"\n🔢 Confusion Matrix:\n{cm_test}")

# Comparar con validation
print(f"\n📈 Comparación:")
print(f"   Validation Accuracy: {validation_accuracy:.4f}")
print(f"   Test Accuracy:       {acc_test:.4f}")
print(f"   Diferencia:           {abs(acc_test - validation_accuracy):.4f}")

# Análisis de diferencias
diferencia = abs(acc_test - validation_accuracy)
if diferencia < 0.05:
    print("   ✅ Diferencia pequeña: modelo generaliza bien")
elif diferencia < 0.10:
    print("   ⚠️ Diferencia moderada: modelo generaliza razonablemente")
else:
    print("   ⚠️ Diferencia grande: posible sobreajuste")

# Análisis detallado por clase
print(f"\n📊 Análisis por clase en Test:")
cr_dict = classification_report(y_test, y_test_pred, output_dict=True)
for clase in ['neg', 'neu', 'pos']:
    if clase in cr_dict:
        print(f"\n   {clase.upper()}:")
        print(f"     Precision: {cr_dict[clase]['precision']:.3f}")
        print(f"     Recall:    {cr_dict[clase]['recall']:.3f}")
        print(f"     F1-score:  {cr_dict[clase]['f1-score']:.3f}")

EVALUACIÓN FINAL EN TEST SET
(Esta es la evaluación final, no debe usarse para ajustar el modelo)

📊 Accuracy en Test: 0.5952 (59.52%)

📋 Classification Report:
              precision    recall  f1-score   support

         neg       0.60      0.66      0.63       140
         neu       0.50      0.54      0.52       140
         pos       0.73      0.59      0.65       140

    accuracy                           0.60       420
   macro avg       0.61      0.60      0.60       420
weighted avg       0.61      0.60      0.60       420


🔢 Confusion Matrix:
[[92 42  6]
 [39 76 25]
 [23 35 82]]

📈 Comparación:
   Validation Accuracy: 0.6381
   Test Accuracy:       0.5952
   Diferencia:           0.0429
   ✅ Diferencia pequeña: modelo generaliza bien

📊 Análisis por clase en Test:

   NEG:
     Precision: 0.597
     Recall:    0.657
     F1-score:  0.626

   NEU:
     Precision: 0.497
     Recall:    0.543
     F1-score:  0.519

   POS:
     Precision: 0.726
     Recall:    0.586
     F1-

## Ahora comparemos esto con un modelo pre-entrenado que el analisis de sentimientos en español, para este caso usaremos pysentmiento.

In [27]:
# ============================================
# COMPARACIÓN CON PYSENTIMIENTO
# ============================================

# Verificar e importar pysentimiento
try:
    from pysentimiento import create_analyzer
    print("✓ pysentimiento importado correctamente")
except ImportError as e:
    print(f"❌ Error al importar pysentimiento: {e}")

# Importar otras librerías necesarias
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import numpy as np

print("✓ Librerías importadas")

✓ pysentimiento importado correctamente
✓ Librerías importadas


In [29]:
# Crear analizador de pysentimiento
print("="*60)
print("CREANDO ANALIZADOR DE PYSENTIMIENTO")
print("="*60)
print("\nCargando modelo pysentimiento...")


analyzer_pysent = create_analyzer(task="sentiment", lang="es")

print("✓ Modelo pysentimiento cargado y listo")

CREANDO ANALIZADOR DE PYSENTIMIENTO

Cargando modelo pysentimiento...
✓ Modelo pysentimiento cargado y listo


In [30]:
# Función para mapear resultados de pysentimiento a nuestras etiquetas
def mapear_pysentimiento(resultado):
    """
    Mapea resultados de pysentimiento a nuestras etiquetas
    pysentimiento devuelve: 'POS', 'NEG', 'NEU'
    Nosotros usamos: 'pos', 'neg', 'neu'
    """
    mapeo = {
        'POS': 'pos',
        'NEG': 'neg',
        'NEU': 'neu'
    }
    return mapeo.get(resultado.output, 'neu')

# Probar la función con un ejemplo
print("Probando función de mapeo...")
texto_prueba = "Esta película es increíble!"
resultado_prueba = analyzer_pysent.predict(texto_prueba)
print(f"Texto: {texto_prueba}")
print(f"Resultado pysentimiento: {resultado_prueba.output}")
print(f"Mapeado: {mapear_pysentimiento(resultado_prueba)}")
print("✓ Función de mapeo funcionando correctamente")

Probando función de mapeo...
Texto: Esta película es increíble!
Resultado pysentimiento: POS
Mapeado: pos
✓ Función de mapeo funcionando correctamente


In [32]:
# Predecir con pysentimiento en el conjunto de TEST
print("="*60)
print("PREDICIENDO CON PYSENTIMIENTO EN TEST SET")
print("="*60)
print("   Estamos procesando", len(X_test), "tweets con pysentimiento")
print("   (pysentimiento usa modelos grandes, es más lento que nuestro modelo)")

# Obtener los textos originales (no procesados) del test set
# Necesitamos los textos originales porque pysentimiento hace su propio preprocesamiento
X_test_original = reviews_balanced.loc[X_test.index if hasattr(X_test, 'index') else range(len(X_test)), 'text']

# Predecir con pysentimiento
y_test_pysent = []
total = len(X_test_original)

print(f"\nProcesando {total} tweets...")
for i, texto in enumerate(X_test_original, 1):
    if i % 50 == 0:  # Mostrar progreso cada 50 tweets
        print(f"  Procesados: {i}/{total} ({i/total*100:.1f}%)")
    
    resultado = analyzer_pysent.predict(str(texto))
    y_test_pysent.append(mapear_pysentimiento(resultado))

y_test_pysent = np.array(y_test_pysent)

print(f"\n✓ Predicciones completadas: {len(y_test_pysent)} tweets procesados")

PREDICIENDO CON PYSENTIMIENTO EN TEST SET
   Estamos procesando 420 tweets con pysentimiento
   (pysentimiento usa modelos grandes, es más lento que nuestro modelo)

Procesando 420 tweets...
  Procesados: 50/420 (11.9%)
  Procesados: 100/420 (23.8%)
  Procesados: 150/420 (35.7%)
  Procesados: 200/420 (47.6%)
  Procesados: 250/420 (59.5%)
  Procesados: 300/420 (71.4%)
  Procesados: 350/420 (83.3%)
  Procesados: 400/420 (95.2%)

✓ Predicciones completadas: 420 tweets procesados


---

<a id="parte-iv"></a>

# 📊 Parte IV: Conclusiones y Comparación Final

En esta sección final, analizaremos los resultados obtenidos con ambos enfoques y proporcionaremos una guía práctica sobre cuándo usar cada método según las necesidades del proyecto.

---


In [33]:
# Evaluar pysentimiento en el conjunto de test
print("="*60)
print("EVALUACIÓN DE PYSENTIMIENTO EN TEST SET")
print("="*60)

# Calcular métricas
acc_pysent = accuracy_score(y_test, y_test_pysent)
cm_pysent = confusion_matrix(y_test, y_test_pysent)
cr_pysent = classification_report(y_test, y_test_pysent)

print(f"\n📊 Accuracy: {acc_pysent:.4f} ({acc_pysent*100:.2f}%)")
print(f"\n📋 Classification Report:\n{cr_pysent}")
print(f"\n🔢 Confusion Matrix:\n{cm_pysent}")

# Guardar para comparar después
pysent_accuracy = acc_pysent

EVALUACIÓN DE PYSENTIMIENTO EN TEST SET

📊 Accuracy: 0.7119 (71.19%)

📋 Classification Report:
              precision    recall  f1-score   support

         neg       0.78      0.84      0.81       140
         neu       0.58      0.66      0.62       140
         pos       0.82      0.64      0.71       140

    accuracy                           0.71       420
   macro avg       0.72      0.71      0.71       420
weighted avg       0.72      0.71      0.71       420


🔢 Confusion Matrix:
[[117  22   1]
 [ 28  93  19]
 [  5  46  89]]


In [34]:
# Comparar ambos modelos
print("="*60)
print("COMPARACIÓN: NAIVE BAYES vs PYSENTIMIENTO")
print("="*60)

print(f"\n📊 ACCURACY:")
print(f"   Naive Bayes (nuestro modelo): {acc_test:.4f} ({acc_test*100:.2f}%)")
print(f"   pysentimiento:                 {pysent_accuracy:.4f} ({pysent_accuracy*100:.2f}%)")
print(f"   Diferencia:                    {abs(acc_test - pysent_accuracy):.4f} ({abs(acc_test - pysent_accuracy)*100:.2f} puntos)")

if acc_test > pysent_accuracy:
    print(f"   ✅ Nuestro modelo es mejor por {((acc_test - pysent_accuracy) * 100):.2f} puntos")
elif pysent_accuracy > acc_test:
    print(f"   ⚠️ pysentimiento es mejor por {((pysent_accuracy - acc_test) * 100):.2f} puntos")
else:
    print(f"   ➡️ Ambos modelos tienen el mismo accuracy")

# Comparar por clase
print(f"\n📊 COMPARACIÓN POR CLASE:")
cr_nb_dict = classification_report(y_test, y_test_pred, output_dict=True)
cr_pysent_dict = classification_report(y_test, y_test_pysent, output_dict=True)

print(f"\n{'Clase':<10} {'Métrica':<12} {'Naive Bayes':<15} {'pysentimiento':<15} {'Mejor':<10}")
print("-" * 70)

for clase in ['neg', 'neu', 'pos']:
    if clase in cr_nb_dict and clase in cr_pysent_dict:
        # Precision
        nb_prec = cr_nb_dict[clase]['precision']
        py_prec = cr_pysent_dict[clase]['precision']
        mejor_prec = "NB" if nb_prec > py_prec else "PY" if py_prec > nb_prec else "="
        print(f"{clase.upper():<10} {'Precision':<12} {nb_prec:<15.3f} {py_prec:<15.3f} {mejor_prec:<10}")
        
        # Recall
        nb_rec = cr_nb_dict[clase]['recall']
        py_rec = cr_pysent_dict[clase]['recall']
        mejor_rec = "NB" if nb_rec > py_rec else "PY" if py_rec > nb_rec else "="
        print(f"{'':<10} {'Recall':<12} {nb_rec:<15.3f} {py_rec:<15.3f} {mejor_rec:<10}")
        
        # F1-score
        nb_f1 = cr_nb_dict[clase]['f1-score']
        py_f1 = cr_pysent_dict[clase]['f1-score']
        mejor_f1 = "NB" if nb_f1 > py_f1 else "PY" if py_f1 > nb_f1 else "="
        print(f"{'':<10} {'F1-score':<12} {nb_f1:<15.3f} {py_f1:<15.3f} {mejor_f1:<10}")
        print("-" * 70)

COMPARACIÓN: NAIVE BAYES vs PYSENTIMIENTO

📊 ACCURACY:
   Naive Bayes (nuestro modelo): 0.5952 (59.52%)
   pysentimiento:                 0.7119 (71.19%)
   Diferencia:                    0.1167 (11.67 puntos)
   ⚠️ pysentimiento es mejor por 11.67 puntos

📊 COMPARACIÓN POR CLASE:

Clase      Métrica      Naive Bayes     pysentimiento   Mejor     
----------------------------------------------------------------------
NEG        Precision    0.597           0.780           PY        
           Recall       0.657           0.836           PY        
           F1-score     0.626           0.807           PY        
----------------------------------------------------------------------
NEU        Precision    0.497           0.578           PY        
           Recall       0.543           0.664           PY        
           F1-score     0.519           0.618           PY        
----------------------------------------------------------------------
POS        Precision    0.726      

## Análisis de conclusiones y ventajas/desventajas

In [35]:
# Análisis de conclusiones
print("="*60)
print("ANÁLISIS DE CONCLUSIONES")
print("="*60)

print("\n📊 RESUMEN DE RESULTADOS:")
print(f"   pysentimiento: 71.19% accuracy (mejor)")
print(f"   Naive Bayes:   59.52% accuracy")
print(f"   Diferencia:     11.67 puntos porcentuales")

print("\n" + "="*60)
print("VENTAJAS Y DESVENTAJAS DE CADA MODELO")
print("="*60)

print("\n✅ NAIVE BAYES (Nuestro Modelo):")
print("   Ventajas:")
print("     ✓ Entrenado específicamente con tus datos de tweets sobre IA")
print("     ✓ Puede aprender patrones específicos del dominio")
print("     ✓ Transparente: puedes ver qué palabras son importantes")
print("     ✓ No requiere internet para funcionar (una vez entrenado)")
print("     ✓ Más rápido en predicción (después del entrenamiento)")
print("     ✓ Usa técnicas básicas que has aprendido en los notebooks")
print("   Desventajas:")
print("     ✗ Requiere datos etiquetados para entrenar")
print("     ✗ Accuracy más bajo: 59.52%")
print("     ✗ Puede sobreajustarse con pocos datos")
print("     ✗ No captura contexto complejo como los transformers")

print("\n✅ PYSENTIMIENTO:")
print("   Ventajas:")
print("     ✓ Pre-entrenado: no requiere entrenamiento")
print("     ✓ Funciona inmediatamente")
print("     ✓ Entrenado con grandes volúmenes de datos")
print("     ✓ Accuracy más alto: 71.19%")
print("     ✓ Mejor en todas las clases (neg, neu, pos)")
print("     ✓ Usa transformers (modelos avanzados)")
print("   Desventajas:")
print("     ✗ No está adaptado a tu dominio específico")
print("     ✗ Requiere más recursos computacionales")
print("     ✗ Más lento en predicción")
print("     ✗ Menos transparente sobre qué palabras usa")
print("     ✗ Requiere descargar modelos grandes")

print("\n" + "="*60)
print("¿CUÁNDO USAR CADA UNO?")
print("="*60)

print("\n🎯 Usa NAIVE BAYES cuando:")
print("   - Tienes datos específicos de tu dominio")
print("   - Necesitas entender qué palabras son importantes")
print("   - Quieres un modelo rápido y ligero")
print("   - Estás aprendiendo NLP (como en este curso)")
print("   - No tienes recursos para modelos grandes")

print("\n🎯 Usa PYSENTIMIENTO cuando:")
print("   - Necesitas la mejor precisión posible")
print("   - No tienes datos etiquetados para entrenar")
print("   - Necesitas resultados inmediatos")
print("   - Tienes recursos computacionales disponibles")
print("   - Trabajas con texto general (no muy específico)")

ANÁLISIS DE CONCLUSIONES

📊 RESUMEN DE RESULTADOS:
   pysentimiento: 71.19% accuracy (mejor)
   Naive Bayes:   59.52% accuracy
   Diferencia:     11.67 puntos porcentuales

VENTAJAS Y DESVENTAJAS DE CADA MODELO

✅ NAIVE BAYES (Nuestro Modelo):
   Ventajas:
     ✓ Entrenado específicamente con tus datos de tweets sobre IA
     ✓ Puede aprender patrones específicos del dominio
     ✓ Transparente: puedes ver qué palabras son importantes
     ✓ No requiere internet para funcionar (una vez entrenado)
     ✓ Más rápido en predicción (después del entrenamiento)
     ✓ Usa técnicas básicas que has aprendido en los notebooks
   Desventajas:
     ✗ Requiere datos etiquetados para entrenar
     ✗ Accuracy más bajo: 59.52%
     ✗ Puede sobreajustarse con pocos datos
     ✗ No captura contexto complejo como los transformers

✅ PYSENTIMIENTO:
   Ventajas:
     ✓ Pre-entrenado: no requiere entrenamiento
     ✓ Funciona inmediatamente
     ✓ Entrenado con grandes volúmenes de datos
     ✓ Accuracy má